# Controlled NN input comparison

Run after notebook 01. Refit eight- and six-input networks under the same recipe. Each variant receives its own validation calibration. This is an exploratory comparison on a previously inspected period.


**Data input:** use the original `5EnvSensor_MayToDec2024_180kRows.csv` directly,
or a long-format extract containing CO2, temperature and humidity. Set your path
once in ignored `config/local.json` (copy `config/local.example.json`).
The loader detects the format, selects sensor `6012002000869`, removes identical
duplicates and sorts complete timestamps. It does not alter the source file or
fill missing values. A CO2-only extract is insufficient. See `data/README.md`.


## 1  Import computation tools

Use the same local project environment as the original experiment.

In [ ]:

# Find the project from its root or the notebooks directory.
from pathlib import Path
import sys
_here=Path.cwd().resolve()
PROJECT_DIR=next((p for p in (_here,*_here.parents) if (p/'src/co2_project').is_dir()),None)
if PROJECT_DIR is None: raise FileNotFoundError('Open this notebook inside the submission project.')
if str(PROJECT_DIR/'src') not in sys.path: sys.path.insert(0,str(PROJECT_DIR/'src'))
from co2_project.project import load_settings, resolve_data_csv, record_run, get_run
from co2_project.data import file_sha256, prepare_environment_csv, make_features, chronological_split, HISTORY_FEATURES
from co2_project.faults import WindowPlanner, write_scenario, inject_fault
from co2_project.models import fit_ols, predict_ols_features, fit_neural, neural_settings
from co2_project.detectors import cusum_series, normalized_cusum, select_cusum_limit
from co2_project.evaluation import evaluate_alarm_events, score_development_events
from functools import partial
SETTINGS=load_settings(PROJECT_DIR)
cusum_series=partial(cusum_series,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
normalized_cusum=partial(normalized_cusum,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])
select_cusum_limit=partial(select_cusum_limit,max_gap_minutes=SETTINGS['MAX_GAP_MINUTES'])

# Import computation tools
import hashlib
import json
import sys
import uuid
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from threadpoolctl import threadpool_limits
from IPython.display import display

print('Dependencies ready; no data have been read and no model has been trained.')
from time import perf_counter

import os
import matplotlib.dates as mdates
from IPython.display import Markdown


## 2  Freeze the comparison protocol

Only remove the two historical CO2 inputs. Reuse magnitudes, reading lengths, fault seeds and split. Every run gets a new directory.

In [ ]:
# Freeze the comparison protocol
# PROJECT_DIR was discovered in the first cell.
BASELINE_DIR=get_run(PROJECT_DIR,'main')
LOCAL_CSV=str(resolve_data_csv(PROJECT_DIR))
TARGET_SENSOR=SETTINGS['TARGET_SENSOR']
MODEL_SEED=SETTINGS['MODEL_SEED']
FAULT_SEED=SETTINGS['FAULT_SEED']
MAX_GAP_MINUTES=SETTINGS['MAX_GAP_MINUTES']
RECOVERY_READINGS=SETTINGS['RECOVERY_READINGS']
THRESHOLD_QUANTILE=SETTINGS['THRESHOLD_QUANTILE']
CUSUM_K_FRACTION=SETTINGS['CUSUM_K_FRACTION']
CUSUM_H_GRID=np.arange(.5,40.5001,.25)
SPIKE_PPM=SETTINGS['SPIKE_PPM']
DRIFT_PPM=SETTINGS['DRIFT_PPM']
BIAS_PPM=SETTINGS['BIAS_PPM']
SPIKE_COUNT=SETTINGS['SPIKE_COUNT']
DRIFT_COUNT=SETTINGS['DRIFT_COUNT']
BIAS_COUNT=SETTINGS['BIAS_COUNT']
DRIFT_READINGS=SETTINGS['DRIFT_READINGS']
BIAS_READINGS=SETTINGS['BIAS_READINGS']
TEST_SEEDS=SETTINGS['TEST_SEEDS']
TEST_MAGNITUDES=SETTINGS['TEST_MAGNITUDES']
TEST_DURATIONS=SETTINGS['TEST_DURATIONS']
result_root=PROJECT_DIR/'results/input_comparison'
run_id=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
RUN_DIR=result_root/run_id
RUN_DIR.mkdir(parents=True,exist_ok=False)
FROZEN_HIDDEN_LAYERS=(16,8)
FROZEN_NN_SETTINGS=neural_settings(MODEL_SEED)
protocol={'study_version':'nn_input_comparison_v2','protocol_revision':2,'date':'2026-10-24',
    'status':'exploratory on the previously inspected test period',
    'hypothesis':'Removing CO2 history prevents injection-induced prediction shifts but may worsen normal prediction and detection.',
    'NN_settings':FROZEN_NN_SETTINGS,'NN_inputs':{'eight':['co2_lag1','co2_lag6','temperature','humidity','hour_sin','hour_cos','weekend','minutes_since_previous'],
        'six':['temperature','humidity','hour_sin','hour_cos','weekend','minutes_since_previous']},
    'same_eligible_rows_and_chronological_split':True,'fit_data':'original training only',
    'point_calibration':'original validation absolute residual Q99 for each NN',
    'CUSUM_calibration':{'k_fraction_of_training_sigma':.1,'h_multipliers':CUSUM_H_GRID.tolist(),
        'validation_budget':'ceil(1% of validation readings)',
        'recipe':'original CUSUM; common quarter-SD grid extension if initial grid cannot meet validation budget',
        'revision_reason':'Initial six-input run failed calibration in 0.5–40.5 SD before any test fault evaluation. Only original validation alerts determine extension.'},
    'test_seeds':TEST_SEEDS,'magnitudes_ppm':TEST_MAGNITUDES,'long_durations_readings':TEST_DURATIONS,
    'default':{'seed':42,'magnitude_ppm':60,'spike_count':20,'long_count':8,'long_duration':24},
    'retained_baseline':BASELINE_DIR.relative_to(PROJECT_DIR).as_posix(),'extra_NN_or_k_search':False,'h_grid_extension_from_validation_only':True,
    'primary_evaluation':'paired additional drift event detections, with all fault types, background burden, reading coverage and delays retained'}
(RUN_DIR/'protocol_before_training.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
baseline_files=sorted(p for p in BASELINE_DIR.rglob('*') if p.is_file())
BASELINE_HASHES={str(p.relative_to(BASELINE_DIR)):hashlib.sha256(p.read_bytes()).hexdigest() for p in baseline_files}
print('New private output:',RUN_DIR)
print('Protocol saved before training. No model selection uses test scores.')


## 3  Read the original data with unchanged preparation

Reuse complete-channel and duplicate checks from the original experiment. No training yet.

In [ ]:
# Read the original data with unchanged preparation
INPUT_CSV = Path(LOCAL_CSV)
if not INPUT_CSV.is_file():
    raise FileNotFoundError(INPUT_CSV)


SOURCE_HASH = file_sha256(INPUT_CSV)
print('Input file:', INPUT_CSV.name)


original, audit = prepare_environment_csv(INPUT_CSV, TARGET_SENSOR)
print('Detected input format:', audit['input_format'])
print('Complete distinct timestamps:', len(original))
print('Identical channel duplicates removed:', audit['identical_duplicates_removed'])
print('Prepared data SHA-256:', audit['prepared_sha256'])
if audit['input_format']=='raw_course_csv':
    print('Raw rows read:', audit['source_rows_read'], '| selected sensor rows:', audit['selected_sensor_rows'],
          '| invalid JSON rows:', audit['invalid_json_rows'])
display(original.head(3))
print('This cell prepares data only; no training has occurred.')


## 4  Keep the same eligible timestamps and chronological split

The six-input variant retains the eight-input eligibility mask: 7373 training, 2458 validation and 2458 test rows.

In [ ]:

# The six-input trial keeps exactly the same eligibility mask as the main study.
prepared=chronological_split(original,MAX_GAP_MINUTES)
original_features=prepared['features']
ELIGIBLE_IDX=prepared['eligible']
TRAIN_IDX,VALIDATION_IDX,TEST_IDX=prepared['train'],prepared['validation'],prepared['test']
split=prepared['split']
MONITORED_IDX=VALIDATION_IDX.append(TEST_IDX)
train_y=original.loc[TRAIN_IDX,'Carbon dioxide']
calibration_observations=original.loc[:VALIDATION_IDX[-1]].copy()
print(f'Train {len(TRAIN_IDX):,}; validation {len(VALIDATION_IDX):,}; test {len(TEST_IDX):,}; excluded {(split=="excluded").sum():,}')

ENVIRONMENT_FEATURES=[f for f in HISTORY_FEATURES if f not in ('co2_lag1','co2_lag6')]
assert len(ENVIRONMENT_FEATURES)==6
assert (len(TRAIN_IDX),len(VALIDATION_IDX),len(TEST_IDX))==(7373,2458,2458)
baseline_report=json.loads((BASELINE_DIR/'run_report.json').read_text(encoding='utf-8'))
# Raw and extracted files have different byte hashes; compare the exact prepared model input.
assert audit['prepared_sha256']==baseline_report['audit']['prepared_sha256']
saved_split=pd.read_csv(BASELINE_DIR/'fault_data/spike/observations.csv',index_col=0,parse_dates=True)
saved_split.index=pd.to_datetime(saved_split.index,utc=True)
pd.testing.assert_index_equal(original.index,saved_split.index)
assert split.tolist()==saved_split['split'].tolist()
# The saved spike copy has corrupted test CO2; its environmental channels and non-test rows must still match.
pd.testing.assert_frame_equal(original[['Temperature','Humidity']],saved_split[['Temperature','Humidity']],check_freq=False,check_names=False)
pd.testing.assert_frame_equal(original.loc[split!='test'],saved_split.loc[split!='test',original.columns],check_freq=False,check_names=False)
display(pd.DataFrame({'feature':HISTORY_FEATURES,'eight_input':True,'six_input':[f in ENVIRONMENT_FEATURES for f in HISTORY_FEATURES]}))


## 5  Define one training and prediction recipe

Both models use the same settings and training-only scaling. Predictors accept observations, not fault labels or reference values.

In [ ]:

networks,metadata,predictors,fit_seconds={},{},{},{}
def fit_variant(key,features):
    """Fit one input variant using the common training-only recipe."""
    fitted=fit_neural(original_features.loc[TRAIN_IDX,features],train_y,settings=FROZEN_NN_SETTINGS)
    def predict(observed):
        """Apply the variant to selected observed features without refitting."""
        available=ELIGIBLE_IDX.intersection(observed.index)
        return fitted.predict(make_features(observed).loc[available])
    networks[key],predictors[key]=fitted.model,predict
    fit_seconds[key]=fitted.seconds
    metadata[key]={**fitted.metadata(),'training_rows':len(TRAIN_IDX),'training_seconds':fitted.seconds}
    print(key,'| inputs:',len(features),'| epochs:',metadata[key]['epochs'],'| warnings:',metadata[key]['warnings'])
    return predict


## 6  Fit the eight-input reproduction control

Weights must exactly match the saved NN before interpreting the new variant.

In [ ]:
# Fit the eight-input reproduction control
predict_eight=fit_variant('history_neural',HISTORY_FEATURES)
assert metadata['history_neural']['weights_sha256']==baseline_report['models']['history_neural']['weights_sha256'], 'Eight-input NN did not reproduce the source run.'
print('Eight-input weight hash matches the existing NN exactly.')


## 7  Fit the six-input NN

Retrain without lag1 and lag6, retaining environment, time and sampling interval. Do not zero out inputs to the old NN.

In [ ]:
# Fit the six-input NN
predict_six=fit_variant('environment_neural',ENVIRONMENT_FEATURES)
assert metadata['environment_neural']['parameters']==257
print('Six-input model fitted with the same hidden layers and training recipe.')


## 8  Measure unmodified prediction and calibrate point limits

Training fits weights; each point limit uses its own unmodified validation absolute-residual Q99. Errors use ppm.

In [ ]:
# Measure unmodified prediction and calibrate point limits
POINT_KEYS=['history_neural','environment_neural']
NAMES={'history_neural':'NN eight inputs','history_neural_cusum':'Eight-input NN + original CUSUM',
       'environment_neural':'NN six inputs','environment_neural_cusum':'Six-input NN + original CUSUM'}
methods={}
calibration_metadata={}
clean_predictions={key:predictors[key](original) for key in POINT_KEYS}
quality_rows=[]
for key in POINT_KEYS:
    p=clean_predictions[key]
    for part,index in [('train',TRAIN_IDX),('validation',VALIDATION_IDX),('test_unmodified',TEST_IDX)]:
        error=original.loc[index,'Carbon dioxide']-p.loc[index]
        quality_rows.append({'method':key,'name':NAMES[key],'part':part,
            'mae_ppm':float(error.abs().mean()),'rmse_ppm':float(np.sqrt((error**2).mean()))})
    validation_prediction=predictors[key](calibration_observations).loc[VALIDATION_IDX]
    residual=calibration_observations.loc[VALIDATION_IDX,'Carbon dioxide']-validation_prediction
    theta=float(residual.abs().quantile(THRESHOLD_QUANTILE))
    methods[key]={'predictor_key':key,'threshold':theta,'kind':'point'}
    calibration_metadata[key]={'threshold_ppm':theta,'source':'unmodified validation Q99',
        'validation_alerts':int((residual.abs()>theta).sum())}
prediction_quality=pd.DataFrame(quality_rows)
display(prediction_quality.round(3))
assert abs(methods['history_neural']['threshold']-32.212719597592525)<1e-9
prediction_quality.to_csv(RUN_DIR/'prediction_quality.csv',index=False)


## 9  Calibrate both CUSUMs with one recipe

Training residuals determine each centre and SD; k is 0.1 SD. Select the first candidate limit on the same SD-multiplier grid meeting the 25-alert validation budget, without fault labels. **Protocol amendment:** the initial six-input run was infeasible within 0.5–40.5 SD. Both variants may now extend in 0.25-SD steps until the budget is met. Only unmodified validation alerts govern extension; the prior failure is documented in reports/protocol_history.md; executed measurement outputs remain private. No test faults had been evaluated at that amendment.

In [ ]:
# Calibrate both CUSUMs with one recipe


def calibrate_cusum(point_key):
    """Apply the unchanged original calibration recipe separately to each trained predictor."""
    started=perf_counter()
    predict=predictors[point_key]
    train_p=predict(original.loc[:TRAIN_IDX[-1]]).loc[TRAIN_IDX]
    train_r=train_y-train_p
    center,sigma=float(train_r.mean()),float(train_r.std(ddof=0))
    if not sigma>0:
        raise ValueError('Training residual SD must be positive.')
    k=CUSUM_K_FRACTION*sigma
    val_p=predict(calibration_observations).loc[VALIDATION_IDX]
    val_r=calibration_observations.loc[VALIDATION_IDX,'Carbon dioxide']-val_p-center
    budget=int(np.ceil((1-THRESHOLD_QUANTILE)*len(VALIDATION_IDX)))
    choice=select_cusum_limit(val_r,k,sigma,THRESHOLD_QUANTILE,CUSUM_H_GRID,allow_extension=True)
    pd.DataFrame(choice['initial_trials']).to_csv(RUN_DIR/f'{point_key}_initial_CUSUM_grid.csv',index=False)
    trials=choice['trials']
    initial_grid_feasible=choice['initial_grid_feasible']
    h,alerts=choice['h'],choice['validation_alerts']
    key=point_key+'_cusum'
    methods[key]={'predictor_key':point_key,'threshold':h,'kind':'cusum'}
    calibration_metadata[key]={'training_residual_center_ppm':center,'training_residual_sd_ppm':sigma,
        'k':k,'h':h,'validation_alerts':alerts,'validation_budget':budget,'trials':trials,
        'initial_0_5_to_40_5_sigma_grid_feasible':initial_grid_feasible,
        'selected_h_in_training_sigma':h/sigma,
        'recipe':'original CUSUM with common validation-only quarter-SD extension if the initial grid is infeasible',
        'calibration_seconds':perf_counter()-started}
    print(NAMES[key],'| k',round(k,3),'| h',round(h,3),'| validation alerts',alerts)

for key in POINT_KEYS:
    calibrate_cusum(key)
EXPERIMENT_KEYS=['history_neural','history_neural_cusum','environment_neural','environment_neural_cusum']
assert abs(calibration_metadata['history_neural_cusum']['h']-127.80559792115086)<1e-8


## 10  Freeze limits before test faults

The eight-input control reproduced the baseline. No extra NN-setting or k search was performed; h uses only the unmodified validation budget. Reset after alarms or gaps over 20 minutes; carry validation state into test.

In [ ]:
# Freeze limits before test faults
frozen={'NN_models':metadata,'calibration':calibration_metadata,
        'split_counts':{'train':len(TRAIN_IDX),'validation':len(VALIDATION_IDX),'test':len(TEST_IDX)},
        'source_sha256':SOURCE_HASH,'protocol':protocol}
(RUN_DIR/'frozen_before_fault_evaluation.json').write_text(json.dumps(frozen,indent=2,allow_nan=False),encoding='utf-8')
threshold_view=[]
for key in EXPERIMENT_KEYS:
    m=methods[key]
    threshold_view.append({'method':key,'name':NAMES[key],'threshold':m['threshold'],
        'unit':'absolute residual ppm' if m['kind']=='point' else 'accumulated centred residual',
        'validation_alerts':calibration_metadata[key]['validation_alerts']})
threshold_table=pd.DataFrame(threshold_view)
display(threshold_table.round(3))
threshold_table.to_csv(RUN_DIR/'thresholds.csv',index=False)


## 11  Reuse identical fault-window planning

Inject test copies only; training and validation stay unchanged. Original references and injection labels are stored separately from detector inputs.

In [ ]:

planner=WindowPlanner(original,split,MAX_GAP_MINUTES,RECOVERY_READINGS)
plan_windows=planner.plan
save_scenario=partial(write_scenario,original=original,split=split,run_dir=RUN_DIR,
                      fault_seed=FAULT_SEED,recovery_readings=RECOVERY_READINGS,source_hash=SOURCE_HASH)
scenarios={}
print('Fault planner ready. No faults have been created.')


## 12  Create default spikes: 20 single-reading offsets of signed 60 ppm

These faults must match the original experiment; both NNs then see the same observations.

In [ ]:
# Create spike on a separate copy. Prediction receives observations only.
spike_events=plan_windows(duration=1,count=SPIKE_COUNT,seed=FAULT_SEED)
spike_observed,spike_offsets=inject_fault(original,spike_events,'spike',SPIKE_PPM)
scenarios['spike']=save_scenario('spike',spike_observed,spike_offsets,spike_events)
display(spike_offsets.loc[spike_events[0]['active']].head().rename('injected_offset_ppm'))


## 13  Create default drift: eight ramps over 24 readings

These faults must match the original experiment; both NNs then see the same observations.

In [ ]:
# Create drift on a separate copy. Prediction receives observations only.
drift_events=plan_windows(duration=DRIFT_READINGS,count=DRIFT_COUNT,seed=FAULT_SEED)
drift_observed,drift_offsets=inject_fault(original,drift_events,'drift',DRIFT_PPM)
scenarios['drift']=save_scenario('drift',drift_observed,drift_offsets,drift_events)
display(drift_offsets.loc[drift_events[0]['active']].head().rename('injected_offset_ppm'))


## 14  Create default bias: eight constant offsets over 24 readings

These faults must match the original experiment; both NNs then see the same observations.

In [ ]:
# Create bias on a separate copy. Prediction receives observations only.
bias_events=plan_windows(duration=BIAS_READINGS,count=BIAS_COUNT,seed=FAULT_SEED)
bias_observed,bias_offsets=inject_fault(original,bias_events,'bias',BIAS_PPM)
scenarios['bias']=save_scenario('bias',bias_observed,bias_offsets,bias_events)
display(bias_offsets.loc[bias_events[0]['active']].head().rename('injected_offset_ppm'))


## 15  Infer before scoring labels and retain the original six

Compute four controlled detectors and verify the eight-input pair reproduces saved results. The overall table retains the six original detectors and appends the two six-input rules. Background alerts are not verified false positives.

In [ ]:
# Infer before scoring labels and retain the original six
inference_timings=[]
def infer(observations):
    """Infer using observations alone; compute each NN prediction once for its two alarm rules."""
    all_p={key:predictors[key](observations) for key in POINT_KEYS}
    inferred={}
    for key in EXPERIMENT_KEYS:
        method=methods[key]
        residual=observations.loc[all_p[method['predictor_key']].index,'Carbon dioxide']-all_p[method['predictor_key']]
        if method['kind']=='cusum':
            info=calibration_metadata[key]
            monitored=MONITORED_IDX.intersection(residual.index)
            score=cusum_series(residual.loc[monitored]-info['training_residual_center_ppm'],info['k'],info['h']).loc[TEST_IDX]
        else:
            score=residual.loc[TEST_IDX]
        alarm=score.abs()>method['threshold']
        inferred[key]=pd.DataFrame({'prediction_ppm':all_p[method['predictor_key']].loc[TEST_IDX],
            'residual_ppm':residual.loc[TEST_IDX],'score':score,'alarm':alarm},index=TEST_IDX)
    return inferred

def evaluate(kind,observations,planned,inferred,original_inferred):
    """Use separate labels only after observation-only inference."""
    summaries,details=[],[]
    for key,points in inferred.items():
        m,rows=evaluate_alarm_events(points.alarm,original_inferred[key].alarm,planned)
        summaries.append({'scenario':kind,'method':key,'name':NAMES[key],**m})
        details.extend({'scenario':kind,'method':key,**d} for d in rows)
    return summaries,details

clean_inference=infer(original)
inferred_by_scenario={}
default_rows,default_events=[],[]
for kind,scenario in scenarios.items():
    saved_obs=pd.read_csv(BASELINE_DIR/f'fault_data/{kind}/observations.csv',index_col=0,parse_dates=True)
    saved_obs.index=pd.to_datetime(saved_obs.index,utc=True)
    pd.testing.assert_frame_equal(scenario['observations'],saved_obs[original.columns],check_names=False,check_freq=False)
    saved_manifest=json.loads((BASELINE_DIR/f'fault_data/{kind}/manifest.json').read_text(encoding='utf-8'))
    for event,saved in zip(scenario['events'],saved_manifest['events']):
        assert event['sign']==saved['sign']
        assert [t.isoformat() for t in event['active']]==saved['active_timestamps_utc']
    output=infer(scenario['observations'])
    inferred_by_scenario[kind]=output
    rows,detail=evaluate(kind,scenario['observations'],scenario['events'],output,clean_inference)
    default_rows+=rows;default_events+=detail
    folder=RUN_DIR/'results'/kind
    folder.mkdir(parents=True,exist_ok=False)
    for key,points in output.items():
        exported=points.copy()
        exported.insert(0,'observed_co2_ppm',scenario['observations'].loc[TEST_IDX,'Carbon dioxide'])
        exported['original_co2_ppm']=original.loc[TEST_IDX,'Carbon dioxide']
        exported['event_id']=scenario['reference'].loc[TEST_IDX,'event_id']
        exported['threshold']=methods[key]['threshold']
        exported.to_csv(folder/f'{key}_points.csv')
controlled_default=pd.DataFrame(default_rows)
controlled_events=pd.DataFrame(default_events)
archived_default=pd.read_csv(BASELINE_DIR/'comparison_all_methods.csv')
compare_columns=[c for c in archived_default.columns if c!='name']
match_keys=['scenario','method']
old_pair=archived_default[archived_default.method.isin(['history_neural','history_neural_cusum'])][compare_columns]
new_pair=controlled_default[controlled_default.method.isin(['history_neural','history_neural_cusum'])][compare_columns]
pd.testing.assert_frame_equal(old_pair.sort_values(match_keys).reset_index(drop=True),new_pair.sort_values(match_keys).reset_index(drop=True),check_dtype=False,atol=1e-9,rtol=1e-9)
combined_default=pd.concat([archived_default,controlled_default[controlled_default.method.str.startswith('environment_')]],ignore_index=True)
assert len(combined_default)==24
controlled_default.to_csv(RUN_DIR/'controlled_default.csv',index=False)
combined_default.to_csv(RUN_DIR/'comparison_all_methods.csv',index=False)
controlled_events.to_csv(RUN_DIR/'event_results.csv',index=False)
display(combined_default[['scenario','method','detected','events','paired_additional_events','unmodified_test_alert_rate']].round(4))


## 16  Compare reading coverage background burden and matched delays

One event hit does not identify every faulty reading. Compare delays only for events detected by both input variants under the same alarm recipe; misses remain missing.

In [ ]:
# Compare reading coverage background burden and matched delays
display(controlled_default[['scenario','method','detected','events','paired_additional_events',
    'reading_recall','unmodified_test_alert_rate','recovery_alert_rate','median_first_alert_delay_minutes']].round(4))
paired_rows=[]
for kind in ['spike','drift','bias']:
    for suffix in ['', '_cusum']:
        left=controlled_events[(controlled_events.scenario==kind)&(controlled_events.method=='history_neural'+suffix)]
        right=controlled_events[(controlled_events.scenario==kind)&(controlled_events.method=='environment_neural'+suffix)]
        paired=left.merge(right,on=['scenario','event_id'],suffixes=('_eight','_six'),validate='one_to_one')
        assert (paired.sign_eight==paired.sign_six).all()
        both=paired.detected_eight & paired.detected_six
        delay=paired.loc[both,'first_alert_delay_minutes_six']-paired.loc[both,'first_alert_delay_minutes_eight']
        paired_rows.append({'scenario':kind,'alarm_recipe':'point' if not suffix else 'original CUSUM',
            'both_detected':int(both.sum()),'eight_only':int((paired.detected_eight&~paired.detected_six).sum()),
            'six_only':int((~paired.detected_eight&paired.detected_six).sum()),
            'neither':int((~paired.detected_eight&~paired.detected_six).sum()),
            'median_six_minus_eight_delay_minutes':float(delay.median()) if len(delay) else None})
matched_delays=pd.DataFrame(paired_rows)
display(matched_delays.round(3))
matched_delays.to_csv(RUN_DIR/'matched_input_variant_delays.csv',index=False)


## 17  Check injection-induced prediction shifts

Fault residual = original residual + offset - prediction shift. Six-input shifts should be zero for CO2-only injections; this does not establish physical CO2 truth.

In [ ]:
# Check injection-induced prediction shifts
decomposition_rows=[]
ending_rows=[]
for kind,scenario in scenarios.items():
    for key in POINT_KEYS:
        p0=clean_predictions[key].loc[TEST_IDX]
        pf=inferred_by_scenario[kind][key].prediction_ppm
        offset=scenario['reference'].loc[TEST_IDX,'offset_ppm']
        original_residual=original.loc[TEST_IDX,'Carbon dioxide']-p0
        shift=pf-p0
        reconstructed=original_residual+offset-shift
        np.testing.assert_allclose(reconstructed,inferred_by_scenario[kind][key].residual_ppm,atol=1e-10,rtol=0)
        if key=='environment_neural':
            np.testing.assert_array_equal(p0.to_numpy(),pf.to_numpy())
        for event in scenario['events']:
            active=event['active']
            for timestamp in active:
                decomposition_rows.append({'scenario':kind,'method':key,'event_id':event['event_id'],'sign':event['sign'],
                    'timestamp_utc':str(timestamp),'original_co2_ppm':float(original.loc[timestamp,'Carbon dioxide']),
                    'observed_co2_ppm':float(scenario['observations'].loc[timestamp,'Carbon dioxide']),
                    'original_prediction_ppm':float(p0.loc[timestamp]),'fault_prediction_ppm':float(pf.loc[timestamp]),
                    'original_residual_ppm':float(original_residual.loc[timestamp]),'injected_offset_ppm':float(offset.loc[timestamp]),
                    'prediction_shift_ppm':float(shift.loc[timestamp]),'fault_residual_ppm':float(reconstructed.loc[timestamp])})
            ending_rows.append(decomposition_rows[-1].copy())
decomposition=pd.DataFrame(decomposition_rows)
ending_decomposition=pd.DataFrame(ending_rows)
display(ending_decomposition[ending_decomposition.scenario=='drift'][
    ['event_id','method','original_prediction_ppm','fault_prediction_ppm','original_residual_ppm',
     'injected_offset_ppm','prediction_shift_ppm','fault_residual_ppm']].round(3))
shift_summary=decomposition.groupby(['scenario','method']).prediction_shift_ppm.agg(['mean','min','max'])
display(shift_summary.round(3))
decomposition.to_csv(RUN_DIR/'prediction_shift_readings.csv',index=False)
ending_decomposition.to_csv(RUN_DIR/'event_end_decomposition.csv',index=False)


## 18  Evaluate the same 63 fault settings

Seeds 7/42/99, magnitudes 20/40/60 ppm and long durations 12/24/48 readings. Verify eight-input sensitivity against the archive and append six-input results without tuning from the test table.

In [ ]:
# Evaluate the same 63 fault settings
sensitivity_rows=[]
sensitivity_event_rows=[]
case_manifests=[]
for kind in ['spike','drift','bias']:
    for seed in TEST_SEEDS:
        for magnitude in TEST_MAGNITUDES:
            for duration in ([1] if kind=='spike' else TEST_DURATIONS):
                count=SPIKE_COUNT if kind=='spike' else DRIFT_COUNT
                planned=plan_windows(duration,count,seed)
                offset=pd.Series(0.0,index=original.index)
                for event in planned:
                    profile=np.arange(1,duration+1)/duration if kind=='drift' else np.ones(duration)
                    offset.loc[event['active']]=event['sign']*magnitude*profile
                observed=original.copy()
                observed['Carbon dioxide']+=offset
                assert (offset.loc[split!='test']==0).all()
                output=infer(observed)
                np.testing.assert_array_equal(output['environment_neural'].prediction_ppm.to_numpy(),
                                              clean_predictions['environment_neural'].loc[TEST_IDX].to_numpy())
                rows,detail=evaluate(kind,observed,planned,output,clean_inference)
                for row in rows:
                    sensitivity_rows.append({'scenario':kind,'seed':seed,'magnitude_ppm':magnitude,'duration_readings':duration,
                        'method':row['method'],'events':row['events'],'detected':row['detected'],
                        'paired_additional_events':row['paired_additional_events'],'reading_recall':row['reading_recall'],
                        'unmodified_test_alert_rate':row['unmodified_test_alert_rate'],
                        'outside_alert_rate':row['outside_fault_and_recovery_alert_rate'],
                        'recovery_alert_rate':row['recovery_alert_rate'],'median_delay_minutes':row['median_first_alert_delay_minutes']})
                for row in detail:
                    sensitivity_event_rows.append({**row,'seed':seed,'magnitude_ppm':magnitude,'duration_readings':duration})
                case_manifests.append({'scenario':kind,'seed':seed,'magnitude_ppm':magnitude,'duration_readings':duration,
                    'events':[{'event_id':e['event_id'],'sign':e['sign'],'active':[t.isoformat() for t in e['active']]} for e in planned]})
controlled_sensitivity=pd.DataFrame(sensitivity_rows)
controlled_sensitivity_events=pd.DataFrame(sensitivity_event_rows)
archived_sensitivity=pd.read_csv(BASELINE_DIR/'test_sensitivity.csv')
sensitivity_keys=['scenario','seed','magnitude_ppm','duration_readings','method']
old=archived_sensitivity[archived_sensitivity.method.isin(['history_neural','history_neural_cusum'])]
new=controlled_sensitivity[controlled_sensitivity.method.isin(['history_neural','history_neural_cusum'])]
pd.testing.assert_frame_equal(old.sort_values(sensitivity_keys).reset_index(drop=True),
    new[old.columns].sort_values(sensitivity_keys).reset_index(drop=True),check_dtype=False,atol=1e-9,rtol=1e-9)
combined_sensitivity=pd.concat([archived_sensitivity,controlled_sensitivity[controlled_sensitivity.method.str.startswith('environment_')]],ignore_index=True)
assert len(case_manifests)==63 and len(combined_sensitivity)==504
combined_sensitivity.to_csv(RUN_DIR/'test_sensitivity.csv',index=False)
controlled_sensitivity.to_csv(RUN_DIR/'controlled_sensitivity.csv',index=False)
controlled_sensitivity_events.to_csv(RUN_DIR/'controlled_sensitivity_events.csv',index=False)
(RUN_DIR/'sensitivity_manifests.json').write_text(json.dumps(case_manifests,indent=2),encoding='utf-8')
pooled=combined_sensitivity[(combined_sensitivity.magnitude_ppm==60)&
    (((combined_sensitivity.scenario=='spike')&(combined_sensitivity.duration_readings==1))|
     ((combined_sensitivity.scenario!='spike')&(combined_sensitivity.duration_readings==24)))].groupby(['scenario','method'])[
    ['events','detected','paired_additional_events']].sum()
display(pooled)
pooled.to_csv(RUN_DIR/'common_conditions_three_seeds.csv')


## 19  Plot both NN predictions around three local events

Black: original, grey: observed, blue: eight inputs, green: six inputs. Red crosses belong to one chosen alarm method. Defaults are spike 1, drift 2, bias 2. This cell changes display only.

In [ ]:
# Plot both NN predictions around three local events
PLOT_EVENT_IDS={'spike':1,'drift':2,'bias':2}
ALARM_METHOD='environment_neural_cusum'
POINTS_BEFORE=4
POINTS_AFTER=8
if ALARM_METHOD not in EXPERIMENT_KEYS:
    raise ValueError('Choose a method from EXPERIMENT_KEYS.')
fig,axes=plt.subplots(3,1,figsize=(12,10),layout='constrained')
for ax,kind in zip(axes,['spike','drift','bias']):
    event=next((e for e in scenarios[kind]['events'] if e['event_id']==PLOT_EVENT_IDS[kind]),None)
    if event is None:raise ValueError('Choose a saved default event ID.')
    active=event['active']
    start,end=TEST_IDX.get_loc(active[0]),TEST_IDX.get_loc(active[-1])
    window=TEST_IDX[max(0,start-POINTS_BEFORE):min(len(TEST_IDX),end+POINTS_AFTER+1)]
    ax.plot(window,original.loc[window,'Carbon dioxide'],color='black',lw=2,label='Original record')
    ax.plot(window,scenarios[kind]['observations'].loc[window,'Carbon dioxide'],color='.6',lw=1.6,label='Observed with faults')
    ax.plot(window,inferred_by_scenario[kind]['history_neural'].loc[window,'prediction_ppm'],color='#246da3',ls='--',label='Eight-input NN prediction')
    ax.plot(window,inferred_by_scenario[kind]['environment_neural'].loc[window,'prediction_ppm'],color='#17836c',ls='-.',label='Six-input NN prediction')
    selected=inferred_by_scenario[kind][ALARM_METHOD].loc[window]
    hits=window[selected.alarm.to_numpy()]
    ax.scatter(hits,scenarios[kind]['observations'].loc[hits,'Carbon dioxide'],color='#c9292c',marker='x',s=45,label=NAMES[ALARM_METHOD]+' alarm',zorder=5)
    if len(active)==1:ax.axvline(active[0],color='#dea02d',label='Injected event')
    else:ax.axvspan(active[0],active[-1],color='#dea02d',alpha=.17,label='Injected event')
    locator=mdates.AutoDateLocator(minticks=4,maxticks=7)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator,tz=timezone.utc))
    ax.set(title=f'{kind.title()} event {event["event_id"]} | active alarms {int(selected.alarm.loc[active].sum())}',ylabel='CO2 (ppm)',xlabel='Time (UTC)')
    ax.grid(alpha=.2)
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='outside upper center',ncol=3,fontsize=8)
(RUN_DIR/'figures').mkdir(exist_ok=True)
fig.savefig(RUN_DIR/'figures/input_variants_local_events.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)


## 20  Compare drift sensitivity of both input variants

Four panels show point and original-CUSUM recipes for eight and six inputs. Cells pool three seeds. Complete eight-detector results are also retained.

In [ ]:
# Compare drift sensitivity of both input variants
aggregate=combined_sensitivity.groupby(['scenario','method','magnitude_ppm','duration_readings'],as_index=False)[
    ['events','detected','paired_additional_events']].sum()
aggregate.to_csv(RUN_DIR/'sensitivity_aggregate.csv',index=False)
fig,axes=plt.subplots(2,2,figsize=(10,7),layout='constrained')
for ax,key in zip(axes.flat,EXPERIMENT_KEYS):
    part=aggregate[(aggregate.scenario=='drift')&(aggregate.method==key)]
    counts=part.pivot(index='magnitude_ppm',columns='duration_readings',values='detected').sort_index()
    totals=part.pivot(index='magnitude_ppm',columns='duration_readings',values='events').reindex(index=counts.index,columns=counts.columns)
    fraction=counts/totals
    ax.imshow(fraction,vmin=0,vmax=1,cmap='Blues',aspect='auto')
    ax.set_xticks(range(len(counts.columns)),[str(int(v)) for v in counts.columns])
    ax.set_yticks(range(len(counts.index)),[f'{v:g}' for v in counts.index])
    ax.set(title=NAMES[key],xlabel='Duration in readings',ylabel='Final magnitude (ppm)')
    for i in range(len(counts.index)):
        for j in range(len(counts.columns)):
            ax.text(j,i,f'{int(counts.iloc[i,j])}/{int(totals.iloc[i,j])}',ha='center',va='center',color='white' if fraction.iloc[i,j]>.58 else '#183044')
fig.savefig(RUN_DIR/'figures/input_variants_drift_grid.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)


## 21  Check consistency and export findings

Check causality, source integrity and validation budgets. Findings use actual results, without assuming six inputs are better.

In [ ]:
# Check consistency and export findings
# Perturb future observations only; all earlier predictions and alarms must stay unchanged.
cutoff=TEST_IDX[len(TEST_IDX)//2]
changed=original.copy()
future=changed.index>cutoff
changed.loc[future,'Carbon dioxide']+=80
changed.loc[future,'Temperature']+=3
changed.loc[future,'Humidity']+=7
future_output=infer(changed)
past=TEST_IDX[TEST_IDX<=cutoff]
for key in EXPERIMENT_KEYS:
    pd.testing.assert_frame_equal(clean_inference[key].loc[past],future_output[key].loc[past],check_exact=True)
assert file_sha256(INPUT_CSV)==SOURCE_HASH
assert {str(p.relative_to(BASELINE_DIR)):hashlib.sha256(p.read_bytes()).hexdigest()
        for p in BASELINE_DIR.rglob('*') if p.is_file()}==BASELINE_HASHES
assert all(calibration_metadata[k]['validation_alerts']<=25 for k in ['history_neural_cusum','environment_neural_cusum'])
checks={'same_split_and_eligible_rows':True,'eight_input_weights_reproduced':True,
    'eight_input_default_metrics_reproduced':True,'eight_input_63_case_metrics_reproduced':True,
    'same_default_observations_and_event_plans':True,
    'six_input_prediction_invariant_default_and_63_cases':True,
    'residual_decomposition_identity':True,'future_changes_do_not_change_past_outputs':True,
    'validation_CUSUM_budget_met':True,'source_csv_and_archived_experiment_unchanged':True,
    'no_test_detection_tuning_or_extra_NN_k_search':True}
def count_default(key,kind):
    row=controlled_default[(controlled_default.method==key)&(controlled_default.scenario==kind)].iloc[0]
    return f'{int(row.detected)}/{int(row.events)}'
def count_pooled(key,kind):
    row=pooled.loc[(kind,key)]
    return f'{int(row.detected)}/{int(row.events)}'
findings=[
    'Eight-input weights, default results and all 63 sensitivity cases reproduce the archive on identical train/validation/test timestamps.',
    'Six-input predictions do not move under CO2-only injection. This follows from the input design and does not establish true CO2 reconstruction.',
    'Default spike/drift/bias: eight-input point '+ '/'.join(count_default('history_neural',k) for k in ['spike','drift','bias'])+
        '; six-input point '+ '/'.join(count_default('environment_neural',k) for k in ['spike','drift','bias'])+'.',
    'Default spike/drift/bias: eight-input original CUSUM '+ '/'.join(count_default('history_neural_cusum',k) for k in ['spike','drift','bias'])+
        '; six-input original CUSUM '+ '/'.join(count_default('environment_neural_cusum',k) for k in ['spike','drift','bias'])+'.',
    'At 60 ppm over 24 long-fault readings across three seeds, drift counts are eight-input CUSUM '+count_pooled('history_neural_cusum','drift')+
        ' and six-input CUSUM '+count_pooled('environment_neural_cusum','drift')+'. Also assess paired added detections, background burden and delays.',
    'Removing CO2 history can worsen normal prediction and widen limits. Unchanged predictions under injection do not by themselves establish improved detection.'
]
for number,text in enumerate(findings,1):display(Markdown(f'**{number}.** {text}'))
timing_summary=pd.DataFrame([{'method':k,'fit_seconds':fit_seconds[k],
    'CUSUM_calibration_seconds':calibration_metadata[k+'_cusum']['calibration_seconds']} for k in POINT_KEYS])
timing_summary.to_csv(RUN_DIR/'fit_calibration_timing.csv',index=False)
report_out={'study_version':'nn_input_comparison_v2','protocol_revision':2,'status':protocol['status'],
    'source_sha256':SOURCE_HASH,'audit':audit,'split_counts':frozen['split_counts'],
    'models':metadata,'calibration':calibration_metadata,'checks':checks,'findings':findings,
    'retained_baseline':BASELINE_DIR.relative_to(PROJECT_DIR).as_posix(),'retained_method_count':6,'combined_method_count':8,
    'sensitivity_cases':63,'combined_sensitivity_rows':len(combined_sensitivity),
    'versions':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__},
    'limits':['Inspected test period; exploratory','Original health unknown; background alerts are not confirmed false positives',
              'Shared measurements and partially overlapping simulations','CO2-only finite faults leave environmental channels unchanged',
              'Same calibration recipe does not guarantee equal observed test alert burdens',
              'One fixed small NN architecture per input variant, not an exhaustive input or architecture search']}
(RUN_DIR/'run_report.json').write_text(json.dumps(report_out,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
(RUN_DIR/'findings.md').write_text('\n\n'.join(f'{i}. {t}' for i,t in enumerate(findings,1)),encoding='utf-8')
display(pd.DataFrame(checks.items(),columns=['check','passed']))
print('Results saved:',RUN_DIR)

record_run(PROJECT_DIR,'input_comparison',RUN_DIR)


This is an input comparison. If six inputs produce larger normal errors, wider limits or more background alerts, retain that outcome to explain the limitation. Further work should follow the complete comparison rather than one selected drift. Local plots use default 60-ppm cases; sensitivity covers all 63 settings.